In [ ]:
%%configure -f
{"vCores": 4, "defaultLakehouse": {"name": "diagnostic", "id": "9d10bce5-1edc-4875-83c4-ac0a98a02775", "workspaceId": "82ad2591-974a-4ad4-ace6-e24879274a4b"}}

# adaptive payload introspection


In [ ]:
import sys, json, time, traceback, uuid, platform as _platform
from pathlib import Path

TIER = "debug"
RUN_ID = time.strftime('%Y%m%d-%H%M%S') + '-' + uuid.uuid4().hex[:6]
FILES_ROOT = Path('/lakehouse/default/Files')
RUN_ROOT = FILES_ROOT / 'fabric_rlm_adaptive_validation' / TIER / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)

summary = {
    'tier': TIER,
    'run_id': RUN_ID,
    'started_at': time.time(),
    'python': _platform.python_version(),
    'stages': [],
    'cases': [],
    'passed': False,
    'error': None,
}
SUMMARY_PATH = RUN_ROOT / 'summary.json'

def write_summary():
    summary['updated_at'] = time.time()
    summary['elapsed_seconds'] = summary['updated_at'] - summary['started_at']
    SUMMARY_PATH.write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')

def stage(name, **fields):
    summary['stages'].append({'stage': name, 't': time.time(), **fields})
    print(f'[stage] {name}', fields if fields else '')
    write_summary()

stage('setup', run_root=str(RUN_ROOT))


In [ ]:
WHEEL_PATH = "/lakehouse/default/Files/fabric_rlm_longcot/wheels/fabric_rlm-0.1.10-py3-none-any.whl"
stage('wheel_check', exists=Path(WHEEL_PATH).exists(),
      size=Path(WHEEL_PATH).stat().st_size if Path(WHEEL_PATH).exists() else 0)
import subprocess
out = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                       '--force-reinstall', '--no-deps', WHEEL_PATH],
                      capture_output=True, text=True)
stage('pip_wheel', rc=out.returncode, stderr_tail=out.stderr[-400:])
if out.returncode != 0:
    summary['error'] = 'wheel install failed'; write_summary()
    raise SystemExit('wheel install failed')

try:
    import dspy
    stage('dspy_present', version=getattr(dspy,'__version__','?'))
except ImportError:
    out2 = subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                            'dspy>=3.1.2'], capture_output=True, text=True)
    stage('pip_dspy', rc=out2.returncode, stderr_tail=out2.stderr[-400:])
    if out2.returncode != 0:
        summary['error'] = 'dspy install failed'; write_summary()
        raise SystemExit('dspy install failed')
    import dspy
    stage('dspy_installed', version=getattr(dspy,'__version__','?'))

for mod in [m for m in list(sys.modules) if m == 'fabric_rlm' or m.startswith('fabric_rlm.')]:
    sys.modules.pop(mod, None)
import fabric_rlm
stage('imported', version=getattr(fabric_rlm, '__version__', '?'))


In [ ]:
from fabric_rlm import RLM, FabricLM

cheap = FabricLM('gpt-4.1-mini', temperature=0.0, cache=False)
strong = FabricLM('gpt-5', reasoning_effort='medium', cache=False)
stage('lms_built')

def always_false_validator(result):
    return False  # force the engine to climb every rung

rlm = RLM(
    signature='question -> answer',
    lm=cheap,
    engine='adaptive',
    adaptive=dict(strong_lm=strong, validator=always_false_validator,
                  max_attempts=2, parallel_rollouts=1),
)
result = rlm.run({'question': 'What is 17 multiplied by 23? Reply with just the number.'})

# Dump everything
def safe(v, n=400):
    try:
        return str(v)[:n]
    except Exception:
        return repr(v)[:n]

dump = {
    'submitted': result.submitted,
    'failure_reason': result.failure_reason,
    'payload_type': type(result.payload).__name__,
    'payload_repr': safe(result.payload, 800),
    'payload_keys': list(result.payload.keys()) if isinstance(result.payload, dict) else None,
    'payload_answer_value': safe(result.payload.get('answer')) if isinstance(result.payload, dict) else None,
    'payload_answer_type': type(result.payload.get('answer')).__name__ if isinstance(result.payload, dict) else None,
    'trajectory_n_turns': len(result.trajectory.turns) if result.trajectory else 0,
    'trajectory_metadata_keys': list((result.trajectory.metadata or {}).keys()) if result.trajectory else [],
}

if result.trajectory and result.trajectory.turns:
    dump['turns_preview'] = []
    for t in result.trajectory.turns[:8]:
        dump['turns_preview'].append({
            'turn': t.turn,
            'turn_type': getattr(t, 'turn_type', '?'),
            'submitted': t.submitted,
            'code': safe(t.code, 300),
            'stdout': safe(t.stdout, 300),
            'response_text': safe(getattr(t, 'response_text', ''), 300),
            'error': safe(getattr(t, 'error', None), 200),
        })

# Adaptive metadata details
adaptive_meta = (result.trajectory.metadata or {}).get('adaptive', {}) if result.trajectory else {}
dump['adaptive_attempts_full'] = adaptive_meta.get('attempts', [])
dump['adaptive_winner_rung'] = adaptive_meta.get('winner_rung')
dump['adaptive_stop_reason'] = adaptive_meta.get('stop_reason')

summary['debug_dump'] = dump
summary['passed'] = True  # mark "ran successfully" for poller
write_summary()
print('PAYLOAD KEYS:', dump['payload_keys'])
print('ANSWER VALUE:', dump['payload_answer_value'])
